# LangPro + Best Models on SICK Trial+Train Entailments

This notebook operationalizes the supervisor request:

> LangPro + best models on SICK entailments from trial+train (1.5K)

In this repository, SICK `trial` is exposed as the loader split `dev` (`SICK_trial.txt`). The local files contain 144 trial entailments and 1299 train entailments, for 1443 entailment examples total, i.e. roughly 1.5K.

The experiment is: run LangPro on each SICK entailment example; when LangPro cannot prove entailment by itself, ask one or more strong LLMs for lexical KB injections; inject those facts into LangPro; save whether the proof becomes correct.

The notebook defaults to a tiny dry run. Change `RUN_EXPERIMENT`, `MAX_PROBLEMS`, and `MODEL_SPECS` before launching an expensive full run.


## 1. Imports

In [1]:
import csv
import importlib.util
import json
import os
import sys
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from pprint import pprint

if sys.version_info < (3, 10):
    raise RuntimeError("Use a Python 3.10+ kernel for this repository.")

# Runtime/path setup. In Colab, set KBPROJECTION_RUNTIME=colab and
# KBPROJECTION_PROJECT_ROOT=/content/drive/MyDrive/kbprojection before this cell,
# or let configure_runtime auto-detect Colab and mount Drive.
_candidate_roots = [
    Path(os.environ.get("KBPROJECTION_PROJECT_ROOT", Path.cwd())).expanduser().resolve(),
    Path.cwd().resolve(),
    Path("/content/drive/MyDrive/kbprojection"),
]
for _candidate_root in _candidate_roots:
    if _candidate_root.exists() and str(_candidate_root) not in sys.path:
        sys.path.insert(0, str(_candidate_root))

import kbprojection
import kbprojection.orchestration as orchestration
from kbprojection import SICKLoader
from kbprojection.runtime import configure_runtime
from kbprojection.models import ExperimentResult, ProblemConfig, TestMode
from kbprojection.utils import get_smallest_problems

RUNTIME_PATHS = configure_runtime()
PROJECT_ROOT = RUNTIME_PATHS.project_root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("kbprojection version:", kbprojection.__version__)
print("Runtime:", RUNTIME_PATHS.runtime)
print("Project root:", PROJECT_ROOT)
print("Working directory:", Path.cwd())


Python: 3.11.13
kbprojection: /Users/jorrytdejong/Documents/RNL paper publication/kbprojection/__init__.py
Working directory: /Users/jorrytdejong/Documents/RNL paper publication


## 2. What To Run

`SICK_trial.txt` is the official trial split, but the codebase calls it `dev`. The full supervisor set is therefore `SICK_train.txt + SICK_trial.txt`, filtered to gold `entailment`.

`MODEL_SPECS` is deliberately editable because provider routing changes. The default first row uses the project default (`gpt-5-mini`). The commented examples reflect the best-performing model names reported in the draft appendix; fill in the provider/model identifiers available through your own API account before a full comparison.


In [3]:
# Safety switch: set True only when ready for live LangPro + LLM calls.
RUN_EXPERIMENT = False

# Use 5 for a smoke test. Use None for all train+trial entailments (~1443 examples).
MAX_PROBLEMS = 5

# Sort by total premise+hypothesis length, matching the paper's shortest-first selection logic.
SHORTEST_FIRST = False

# Repository/codebase settings.
DATA_DIR = RUNTIME_PATHS.data_dir / "sick"
SICK_SPLITS = ["train", "dev"]  # dev == SICK_trial.txt in SICKLoader
PROMPT_STYLE = "icl"
TEST_MODE = TestMode.BOTH
RUN_ABLATION = False
POST_PROCESS = True
VERBOSE = False

# LangPro endpoint used by kbprojection.langpro.langpro_api_call.
LANGPRO_ENDPOINT = "https://langpro-annotator.hum.uu.nl/langpro-api/prove/"

# Edit this list for the actual providers/models you want to compare.
# provider can be: openai, openrouter, gemini, or claude, as implemented in kbprojection.llm.GenericAIClient.
MODEL_SPECS = [
    {"run_label": "gpt-5-mini", "provider": "openai", "model": "gpt-5-mini"},
    # Examples to adapt if available in your provider account:
    # {"run_label": "claude-opus-4.5", "provider": "openrouter", "model": "anthropic/claude-opus-4.5"},
    # {"run_label": "claude-sonnet-4.5", "provider": "openrouter", "model": "anthropic/claude-sonnet-4.5"},
    # {"run_label": "gemini-3-flash-preview", "provider": "openrouter", "model": "google/gemini-3-flash-preview"},
]

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
CACHE_ROOT = RUNTIME_PATHS.cache_root / "sick_trial_train_best_models"
RESULTS_DIR = RUNTIME_PATHS.results_dir
RUN_NAME = f"sick_trial_train_entailments_{RUN_STAMP}"

print("RUN_EXPERIMENT:", RUN_EXPERIMENT)
print("MAX_PROBLEMS:", MAX_PROBLEMS)
print("Models:")
pprint(MODEL_SPECS)


RUN_EXPERIMENT: False
MAX_PROBLEMS: 5
Models:
[{'model': 'gpt-5-mini', 'provider': 'openai', 'run_label': 'gpt-5-mini'}]


## 3. Preflight Checks

This cell checks installed packages, relevant API keys, output directories, and the LangPro endpoint patch. It does not call LangPro or an LLM.


In [4]:
def has_package(name: str) -> bool:
    try:
        return importlib.util.find_spec(name) is not None
    except ModuleNotFoundError:
        return False

provider_env = {
    "openai": "OPENAI_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "claude": "ANTHROPIC_API_KEY",
}

print("Package checks:")
for package in ["requests", "pydantic", "nltk", "openai", "anthropic", "google.genai"]:
    print(f"  {package:14s}", "OK" if has_package(package) else "missing")

print("\nProvider keys needed by MODEL_SPECS:")
for provider in sorted({m["provider"] for m in MODEL_SPECS}):
    env_name = provider_env.get(provider)
    if env_name:
        print(f"  {provider:10s} {env_name:20s}", "set" if os.environ.get(env_name) else "not set")
    else:
        print(f"  {provider:10s} unknown provider")

CACHE_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("\nCache root:", CACHE_ROOT.resolve())
print("Results dir:", RESULTS_DIR.resolve())

# Patch the orchestration module so process_single_problem uses the chosen endpoint.
def langpro_api_call_with_endpoint(premises, hypothesis, **kwargs):
    kwargs.setdefault("endpoint", LANGPRO_ENDPOINT)
    return original_langpro_api_call(premises, hypothesis, **kwargs)

orchestration.langpro_api_call = langpro_api_call_with_endpoint
print("LangPro endpoint patched to:", LANGPRO_ENDPOINT)


Package checks:
  requests       OK
  pydantic       OK
  nltk           OK
  openai         OK
  anthropic      OK
  google.genai   OK

Provider keys needed by MODEL_SPECS:
  openai     OPENAI_API_KEY       set

Cache root: /Users/jorrytdejong/Documents/RNL paper publication/experiment_cache/sick_trial_train_best_models
Results dir: /Users/jorrytdejong/Documents/RNL paper publication/experiment_results
LangPro endpoint patched to: https://langpro-annotator.hum.uu.nl/langpro-api/prove/


## 4. Build The SICK Trial+Train Entailment Set

In [5]:
loader = SICKLoader(data_dir=DATA_DIR)
loader.load(splits=SICK_SPLITS)

all_entailments = []
counts_by_split = {}
for split in SICK_SPLITS:
    split_examples = [
        p for p in loader.iter_problems(split=split, label_filter={"entailment"})
    ]
    counts_by_split[split] = len(split_examples)
    all_entailments.extend(split_examples)

if SHORTEST_FIRST:
    all_entailments.sort(key=lambda p: sum(len(s.split()) for s in p.premises) + len(p.hypothesis.split()))

selected_problems = all_entailments if MAX_PROBLEMS is None else all_entailments[:MAX_PROBLEMS]

print("Entailments by split:", counts_by_split)
print("Total trial+train entailments:", len(all_entailments))
print("Selected for this run:", len(selected_problems))
print("First selected examples:")
for p in selected_problems[:5]:
    length = sum(len(s.split()) for s in p.premises) + len(p.hypothesis.split())
    print(f"  {p.split:5s} id={p.id:>5s} len={length:2d} P={p.premises[0]} H={p.hypothesis}")


[SICKLoader] Loaded 4500 problems from train.
[SICKLoader] Loaded 500 problems from dev.
Entailments by split: {'train': 1299, 'dev': 144}
Total trial+train entailments: 1443
Selected for this run: 5
First selected examples:
  train id=    3 len=23 P=The young boys are playing outdoors and the man is smiling nearby H=The kids are playing outdoors near a man with a smile
  train id=   30 len=28 P=A man with a jersey is dunking the ball at a basketball game H=The ball is being dunked by a man with a jersey at a basketball game
  train id=   44 len=17 P=Two young women are sparring in a kickboxing fight H=Two women are sparring in a kickboxing match
  train id=   55 len=14 P=Three boys are jumping in the leaves H=Three kids are jumping in the leaves
  train id=   77 len=26 P=People wearing costumes are gathering in a forest and are looking in the same direction H=Masked people are looking in the same direction in a forest


## 5. Core Runner

The runner uses the codebase's `process_single_problem`, so each example follows the same pipeline as the rest of the project:

1. LangPro without extra KB.
2. If baseline is already correct, stop.
3. If baseline is neutral/unsolved, call the configured LLM.
4. Filter/post-process candidate `isa_wn` / `disj` relations.
5. Re-run LangPro with raw and/or filtered KB depending on `TEST_MODE`.
6. Cache one JSON result per `(model, split, problem id)`.


In [6]:
def safe_problem_id(problem_id: str) -> str:
    return str(problem_id).replace("/", "_").replace("#", "_").replace(" ", "_")


def cache_path_for(run_label: str, problem) -> Path:
    return CACHE_ROOT / run_label / f"sick_{problem.split}_{safe_problem_id(problem.id)}.json"


def load_cached_result(path: Path):
    if not path.exists():
        return None
    try:
        return ExperimentResult.model_validate_json(path.read_text(encoding="utf-8"))
    except Exception as exc:
        print("Could not load cache", path, "->", exc)
        return None


def config_for_model(spec: dict) -> ProblemConfig:
    return ProblemConfig(
        llm_provider=spec["provider"],
        model=spec["model"],
        prompt_style=PROMPT_STYLE,
        post_process=POST_PROCESS,
        test_mode=TEST_MODE,
        run_ablation=RUN_ABLATION,
        verbose=VERBOSE,
    )


def run_one_model(spec: dict, problems: list) -> list[ExperimentResult]:
    run_label = spec["run_label"]
    cache_dir = CACHE_ROOT / run_label
    cache_dir.mkdir(parents=True, exist_ok=True)
    config = config_for_model(spec)
    model_results = []

    print(f"\n=== {run_label} ({spec['provider']} / {spec['model']}) ===")
    for index, problem in enumerate(problems, start=1):
        path = cache_path_for(run_label, problem)
        cached = load_cached_result(path)
        if cached is not None:
            model_results.append(cached)
            print(f"[{index:4d}/{len(problems)}] {problem.split}/{problem.id}: cached {cached.final_status}")
            continue

        print(f"[{index:4d}/{len(problems)}] {problem.split}/{problem.id}: running")
        result = process_single_problem(problem, config=config, cache_file=path)
        model_results.append(result)
        print("    status:", result.final_status, "no_kb:", result.pred_no_kb, "with_kb:", result.pred_with_kb)

    return model_results


## 6. Launch Or Load Results

In [7]:
all_results: dict[str, list[ExperimentResult]] = {}

if not RUN_EXPERIMENT:
    print("Experiment not launched. Set RUN_EXPERIMENT = True after checking MODEL_SPECS.")
    print("Trying to load any existing cached results for the selected models/problems instead.")
    for spec in MODEL_SPECS:
        loaded = []
        for problem in selected_problems:
            cached = load_cached_result(cache_path_for(spec["run_label"], problem))
            if cached is not None:
                loaded.append(cached)
        all_results[spec["run_label"]] = loaded
        print(spec["run_label"], "cached results loaded:", len(loaded))
else:
    for spec in MODEL_SPECS:
        all_results[spec["run_label"]] = run_one_model(spec, selected_problems)


Experiment not launched. Set RUN_EXPERIMENT = True after checking MODEL_SPECS.
Trying to load any existing cached results for the selected models/problems instead.
gpt-5-mini cached results loaded: 0


## 7. Summaries

In [8]:
def result_row(run_label: str, result: ExperimentResult) -> dict:
    problem = result.problem
    return {
        "run_label": run_label,
        "dataset": problem.dataset,
        "split": problem.split,
        "id": problem.id,
        "gold": problem.gold_label.value,
        "premise": " || ".join(problem.premises),
        "hypothesis": problem.hypothesis,
        "pred_no_kb": result.pred_no_kb.value if result.pred_no_kb else None,
        "pred_with_raw_kb": result.pred_with_raw_kb.value if result.pred_with_raw_kb else None,
        "pred_with_kb": result.pred_with_kb.value if result.pred_with_kb else None,
        "final_status": result.final_status.value if hasattr(result.final_status, "value") else str(result.final_status),
        "fixed_by": result.fixed_by,
        "kb_raw": " | ".join(result.kb_raw or []),
        "kb_filtered": " | ".join(result.kb_filtered or []),
        "essential_kb": " | ".join(result.essential_kb or []),
    }


def summarize_model(results: list[ExperimentResult]) -> dict:
    n = len(results)
    no_kb_correct = sum(r.pred_no_kb == r.problem.gold_label for r in results)
    with_kb_correct = sum(
        (r.pred_with_kb == r.problem.gold_label) or (r.pred_with_raw_kb == r.problem.gold_label) or (r.pred_no_kb == r.problem.gold_label)
        for r in results
    )
    fixed = sum(str(r.final_status).endswith("fixed") or str(r.final_status) == "fixed" or r.fixed_by for r in results)
    statuses = Counter(r.final_status.value if hasattr(r.final_status, "value") else str(r.final_status) for r in results)
    return {
        "n": n,
        "no_kb_correct": no_kb_correct,
        "no_kb_accuracy": no_kb_correct / n if n else 0,
        "with_kb_correct_or_already": with_kb_correct,
        "with_kb_accuracy_or_already": with_kb_correct / n if n else 0,
        "newly_fixed_by_kb": fixed,
        "newly_fixed_rate": fixed / n if n else 0,
        "statuses": dict(statuses),
    }

summary = {run_label: summarize_model(results) for run_label, results in all_results.items()}
pprint(summary)


{'gpt-5-mini': {'n': 0,
                'newly_fixed_by_kb': 0,
                'newly_fixed_rate': 0,
                'no_kb_accuracy': 0,
                'no_kb_correct': 0,
                'statuses': {},
                'with_kb_accuracy_or_already': 0,
                'with_kb_correct_or_already': 0}}


## 8. Save CSV And JSONL

In [ ]:
flat_rows = []
for run_label, results in all_results.items():
    flat_rows.extend(result_row(run_label, r) for r in results)

csv_path = RESULTS_DIR / f"{RUN_NAME}.csv"
jsonl_path = RESULTS_DIR / f"{RUN_NAME}.jsonl"
summary_path = RESULTS_DIR / f"{RUN_NAME}_summary.json"

if flat_rows:
    fieldnames = list(flat_rows[0].keys())
    with csv_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(flat_rows)

    with jsonl_path.open("w", encoding="utf-8") as f:
        for row in flat_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

    print("Saved:")
    print("  CSV:    ", csv_path.resolve())
    print("  JSONL:  ", jsonl_path.resolve())
    print("  Summary:", summary_path.resolve())
else:
    print("No rows to save yet.")


## 9. Inspect Fixed Examples

In [ ]:
for run_label, results in all_results.items():
    print("\n===", run_label, "fixed examples ===")
    fixed_results = [r for r in results if r.fixed_by]
    if not fixed_results:
        print("No fixed examples in the loaded/selected results.")
        continue
    for r in fixed_results[:10]:
        print("-" * 100)
        print(f"{r.problem.split}/{r.problem.id}")
        print("P:", " || ".join(r.problem.premises))
        print("H:", r.problem.hypothesis)
        print("no-KB:", r.pred_no_kb, "raw:", r.pred_with_raw_kb, "filtered:", r.pred_with_kb, "fixed_by:", r.fixed_by)
        print("KB raw:", r.kb_raw)
        print("KB filtered:", r.kb_filtered)


## 10. Full-Run Checklist

Before the full run, make sure:

- `MODEL_SPECS` contains the actual provider/model IDs your API account supports.
- The needed API keys are visible in the notebook kernel environment.
- `MAX_PROBLEMS = None` if you really want all 1443 entailments.
- `RUN_EXPERIMENT = True`.
- You expect many remote calls: up to about `1443 x number_of_models`, plus extra LangPro calls for raw/filtered KB and optional ablation.
